<a href="https://colab.research.google.com/github/KalinaMarkova/deep_learning_course_project/blob/main/02_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import os
import glob
import shutil
from tqdm import tqdm
import json
from google.colab import drive
from datasets import Dataset
from transformers import AutoTokenizer

In [25]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import PunktSentenceTokenizer

In [8]:
!pip install -q datasets

In [9]:
!pip install -q --upgrade transformers tokenizers

# Fine-Grained Analysis of Propaganda in News Articles
## Notebook 02: Data Preprocessing & Tokenization

In this notebook, we will load our  dataset and prepare it for the neural network. Since we are doing **Span Identification** (Subtask 1) and **Technique Classification** (Subtask 2), we need to format the text so a Transformer model (like RoBERTa or BERT) can understand it.

We will prepare two datasets to run two experiments:

1. A classification model will attempt to simulationsly solve both subtasks, guessing fro each token whether it is non-propaganda or a specific type of proganda.

2. Two distinct models, with the first one finding only  the boundaries of the propaganda text, while the second model will carry out the categorization of the text.

In [10]:
drive.mount('/content/drive')

csv_path = "/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/propaganda_train_cleaned.csv"

df = pd.read_csv(csv_path)

df.head()

Mounted at /content/drive


,article_id,technique,start,end,snippet,span_length,full_text_length
0,999000870,Repetition,3812,3831,migrant caravan hea,19,4597
1,111111117,Causal_Oversimplification,671,753,the delay signaled the White House was having ...,82,1064
2,780619695,Repetition,1538,1554,How inconvenient,16,6684
3,780619695,Repetition,1728,1744,How inconvenient,16,6684
4,780619695,Repetition,2018,2034,How inconvenient,16,6684


Now let's initialize the **Fast Tokenizer**. In the SemEval dataset, the propaganda labels are based on character indices. For example, the dataset tells us that a "Loaded Language" snippet exists from character 45 to character 60 in the raw text. However, RoBERTa does not look at characters, it looks at tokens. A standard tokenizer will chop up the text but forget where those tokens originally came from. The Fast Tokenizer returns a special dictionary called **offset_mapping**. This creates a  map linking every single generated token back to its exact character start and end positions in the original text. For this task, we will use RoBERTa.

Generally, a tokenizer first reads the raw string and does a basic split, usually by spaces and punctuation. Then it performs subword splitting, breaksing complex or rare words down into smaller, recognizable chunks. After that the tokenizer looks up every single piece in its massive, pre-trained dictionary and swaps the text chunk for its corresponding ID number. The tokenizer then  injects special structural tokens into the sequence. For example RoBERTa adds `<s>` (Start of Sequence) and `</s>` (End of Sequence). Finally, the tokenizer performs truncation and padding. Neural networks require data to be fed in perfectly uniform batches, like a perfect rectangular matrix. As human sentences are of different lengths, the tokenizer fixes this by cutting off a sentence if it is too long or if it is too short, it fills the rest of the sequence with empty padding tokens (usually ID 1) until it hits the required length.

Let's load the tokenizer and demonstrate what it does.

In [11]:
# 1. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('roberta-base', use_fast=True)

# 2. Process the text
sample_text = "The deeply-corrupt politician lied."
encoded = tokenizer(sample_text)

# 3. Print the results
print(f"Original Text: {sample_text}\n")

# Show the raw mathematical IDs the model actually sees
print(f"Token IDs: {encoded['input_ids']}\n")

# Show the human-readable subword translation
print(f"Tokens: {tokenizer.convert_ids_to_tokens(encoded['input_ids'])}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Original Text: The deeply-corrupt politician lied.

Token IDs: [0, 133, 4814, 12, 7215, 14709, 8676, 15005, 4, 2]

Tokens: ['<s>', 'The', 'Ġdeeply', '-', 'cor', 'rupt', 'Ġpolitician', 'Ġlied', '.', '</s>']


**The Ġ symbol** represents a space. Ġdeeply has one, but '-' does not have any. That is how the tokenizer knows there was no space before the hyphen.

**cor and rupt** are an example of subword tokenization. The word "corrupt" was not heavily prioritized in the tokenizer's base vocabulary, so it split it into two common subwords. When the model reads this, it pieces the meaning back together.

Now let's demonstrate the "Offset Mapping" on the same sample.

In [12]:
encoded = tokenizer(sample_text, return_offsets_mapping=True)
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
offsets = encoded['offset_mapping']

print("How the Tokenizer maps Tokens back to Characters:")
print(f"{'Token':<15} {'Char Start':<15} {'Char End':<15}")
print("-" * 45)

for token, offset in zip(tokens, offsets):
    print(f"{token:<15} {offset[0]:<15} {offset[1]:<15}")

How the Tokenizer maps Tokens back to Characters:
Token           Char Start      Char End       
---------------------------------------------
<s>             0               0              
The             0               3              
Ġdeeply         4               10             
-               10              11             
cor             11              14             
rupt            14              18             
Ġpolitician     19              29             
Ġlied           30              34             
.               34              35             
</s>            0               0              


Since our CSV only contains the start/end indexes and the labels, we need to load the original article texts to align the tokens. We will group our annotations by article, read the corresponding text file, and run our alignment function.

In [13]:
drive_path = "/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/datasets-v2.tgz"
local_path = "/content/datasets-v2.tgz"

# Retrieve archive from Drive
if os.path.exists(drive_path):
    shutil.copy(drive_path, local_path)
else:
    raise FileNotFoundError(f"Archive not found at {drive_path}. Please check the path.")

# Extract the archive
!tar -xzf {local_path} -C /content/

# Verify extraction
text_files = glob.glob("/content/**/*.txt", recursive=True)
print(f"Extraction complete. Found {len(text_files)} text files in total.")

Extraction complete. Found 1824 text files in total.


In [23]:
# Build Article File Map
text_files = glob.glob("/content/**/*.txt", recursive=True)
article_files = [f for f in text_files if "article" in os.path.basename(f).lower()]

article_file_map = {}
for filepath in article_files:
    filename = os.path.basename(filepath)
    art_id = filename.replace("article", "").replace(".txt", "")
    article_file_map[art_id] = filepath

# Group your annotations dataframe (Ensure `df` with annotations is loaded before this step)
grouped_annotations = df.groupby('article_id')
print(f"Mapped {len(article_file_map)} article text files.")

Mapped 446 article text files.


Because our neural network predicts a label for every single token, we cannot just label both **cor and rupt** as "Loaded Language". If two separate "Loaded Language" phrases appear right next to each other, the model would not know where one ends and the other begins. We need to introduce the standard formatting used for NLP Token Classification, known as BIO Tagging (Begin, Inside, Outside).


**B-Technique (Begin)**: The very first token of a propaganda phrase.

**I-Technique (Inside)**: Any subsequent tokens that belong to the same phrase.

**O (Outside)**: Normal text that is not propaganda.

In [24]:
# --- Alignment Functions ---
def align_tokens_and_labels(raw_text, annotations, tokenizer):
    char_labels = ["O"] * len(raw_text)
    for ann in annotations:
        start, end, technique = ann['start'], ann['end'], ann['technique']
        if start < 0 or end > len(raw_text):
            continue
        char_labels[start] = f"B-{technique}"
        for i in range(start + 1, end):
            char_labels[i] = f"I-{technique}"

    encoded = tokenizer(raw_text, return_offsets_mapping=True, truncation=True, max_length=512)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
    offsets = encoded['offset_mapping']
    input_ids = encoded['input_ids']

    token_labels = [
        "O" if offset == (0, 0) else char_labels[offset[0]]
        for offset in offsets
    ]
    return tokens, token_labels, input_ids

def align_tokens_and_labels_exp2(raw_text, annotations, tokenizer):
    char_labels = ["O"] * len(raw_text)
    for ann in annotations:
        start, end = ann['start'], ann['end']
        if start < 0 or end > len(raw_text):
            continue
        char_labels[start] = "B-Propaganda"
        for i in range(start + 1, end):
            char_labels[i] = "I-Propaganda"

    encoded = tokenizer(raw_text, return_offsets_mapping=True, truncation=True, max_length=512)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
    offsets = encoded['offset_mapping']
    input_ids = encoded['input_ids']

    token_labels = [
        "O" if offset == (0, 0) else char_labels[offset[0]]
        for offset in offsets
    ]
    return tokens, token_labels, input_ids

# --- Label Mappings ---
# Exp 1 (29 categories)
all_unique_techniques = df['technique'].unique().tolist()
exp1_labels_list = ['O'] + [f"B-{t}" for t in all_unique_techniques] + [f"I-{t}" for t in all_unique_techniques]
exp1_labels_list = sorted(list(set(exp1_labels_list)))
if 'O' in exp1_labels_list:
    exp1_labels_list.remove('O')
    exp1_labels_list = ['O'] + exp1_labels_list

label2id = {label: i for i, label in enumerate(exp1_labels_list)}

# Exp 2 (3 categories)
exp2_unique_labels = ['O', 'B-Propaganda', 'I-Propaganda']
exp2_label2id = {label: i for i, label in enumerate(exp2_unique_labels)}

Now let's add two functions for each experiment that will ensure that truncuation happens without cutting a propaganda snippet in two and losing it as a learning example for our model. This means that every B- tag will be followed by its complete sequence of I- tags within the same sample.

In [27]:
# --- EXPERIMENT 1 FUNCTION (29 Classes / Specific Techniques) ---
def align_for_exp1_29_classes(raw_text, annotations, tokenizer):
    char_labels = ["O"] * len(raw_text)
    for ann in annotations:
        start, end, technique = ann['start'], ann['end'], ann['technique']
        if start < 0 or end > len(raw_text):
            continue
        char_labels[start] = f"B-{technique}"
        for i in range(start + 1, end):
            char_labels[i] = f"I-{technique}"

    encoded = tokenizer(raw_text, return_offsets_mapping=True, truncation=True, max_length=512)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
    offsets = encoded['offset_mapping']
    input_ids = encoded['input_ids']

    token_labels = [
        "O" if offset == (0, 0) else char_labels[offset[0]]
        for offset in offsets
    ]
    return tokens, token_labels, input_ids


# --- EXPERIMENT 2 FUNCTION (3 Classes / General Propaganda) ---
def align_for_exp2_3_classes(raw_text, annotations, tokenizer):
    char_labels = ["O"] * len(raw_text)
    for ann in annotations:
        start, end = ann['start'], ann['end']
        if start < 0 or end > len(raw_text):
            continue
        char_labels[start] = "B-Propaganda"
        for i in range(start + 1, end):
            char_labels[i] = "I-Propaganda"

    encoded = tokenizer(raw_text, return_offsets_mapping=True, truncation=True, max_length=512)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
    offsets = encoded['offset_mapping']
    input_ids = encoded['input_ids']

    token_labels = [
        "O" if offset == (0, 0) else char_labels[offset[0]]
        for offset in offsets
    ]
    return tokens, token_labels, input_ids

No let's tokenize the texts, map the character-level annotations to RoBERTa's token IDs and construct the final datasets for model training.

In [28]:
exp1_tokens, exp1_labels, exp1_input_ids = [], [], []

print("Processing Experiment 1 (29-Class Sentence Level)...")
for article_id, group in tqdm(grouped_annotations):
    str_art_id = str(article_id)
    if str_art_id not in article_file_map:
        continue

    with open(article_file_map[str_art_id], "r", encoding="utf-8") as f:
        raw_text = f.read()

    annotations = group[['start', 'end', 'technique']].to_dict('records')

    # Sentence-level processing with original align_tokens_and_labels
    s_tokens, s_labels, s_ids = process_article_by_sentences(
        raw_text, annotations, tokenizer, align_tokens_and_labels
    )

    exp1_tokens.extend(s_tokens)
    exp1_labels.extend(s_labels)
    exp1_input_ids.extend(s_ids)

# Encode Labels to Integer IDs
exp1_label_ids = [[label2id[l] for l in labels] for labels in exp1_labels]

# Create & Save Dataset
exp1_hf_dataset = Dataset.from_dict({
    "input_ids": exp1_input_ids,
    "tokens": exp1_tokens,
    "labels": exp1_label_ids
})

exp1_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda/exp1_joint_29labels_sentence_dataset'
exp1_hf_dataset.save_to_disk(exp1_path)
print(f"Exp 1 Saved successfully ({len(exp1_hf_dataset)} sentences) to: {exp1_path}")

Processing Experiment 1 (29-Class Sentence Level)...


100%|██████████| 357/357 [00:11<00:00, 31.50it/s]


Saving the dataset (0/1 shards):   0%|          | 0/15028 [00:00<?, ? examples/s]

Exp 1 Saved successfully (15028 sentences) to: /content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda/exp1_joint_29labels_sentence_dataset


In [29]:
exp2_tokens, exp2_labels, exp2_input_ids = [], [], []

print("Processing Experiment 2 (3-Class Sentence Level)...")
for article_id, group in tqdm(grouped_annotations):
    str_art_id = str(article_id)
    if str_art_id not in article_file_map:
        continue

    with open(article_file_map[str_art_id], "r", encoding="utf-8") as f:
        raw_text = f.read()

    annotations = group[['start', 'end', 'technique']].to_dict('records')

    # Sentence-level processing with align_tokens_and_labels_exp2
    s_tokens, s_labels, s_ids = process_article_by_sentences(
        raw_text, annotations, tokenizer, align_tokens_and_labels_exp2
    )

    exp2_tokens.extend(s_tokens)
    exp2_labels.extend(s_labels)
    exp2_input_ids.extend(s_ids)

# Encode Labels to Integer IDs
exp2_label_ids = [[exp2_label2id[l] for l in labels] for labels in exp2_labels]

# Create & Save Dataset
exp2_hf_dataset = Dataset.from_dict({
    "input_ids": exp2_input_ids,
    "tokens": exp2_tokens,
    "labels": exp2_label_ids
})

exp2_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda/exp2_span_3labels_sentence_dataset'
exp2_hf_dataset.save_to_disk(exp2_path)
print(f"Exp 2 Saved successfully ({len(exp2_hf_dataset)} sentences) to: {exp2_path}")

Processing Experiment 2 (3-Class Sentence Level)...


100%|██████████| 357/357 [00:22<00:00, 15.66it/s]


Saving the dataset (0/1 shards):   0%|          | 0/15028 [00:00<?, ? examples/s]

Exp 2 Saved successfully (15028 sentences) to: /content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda/exp2_span_3labels_sentence_dataset


We have now saved our dataset into a standardized Hugging Face Dataset object. This format is optimized for speed and memory, which RoBERTa requires for training.